# Create helper methods to:
 1. Load the CoDocGen model to generate documentation for a function
 2. Method that uses the CoDocModel model to generate documentation of the given code (as string)
 3. Load the the-stack dataset and extract only the C++ and python code from it.
 4. load the github/tree-sitter to create abstract syntx trees of given code and lanugage
 5. Create ASTs for the given code.

# Install all the necessary packages

In [1]:
pip install transformers torch accelerate  datasets  tree-sitter tree-sitter-languages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.4/635.4 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 71.0 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

# Try to retrieve the Hugging Face API key from Colab's secrets manager
HUGGING_FACE_KEY = userdata.get('HUGGING_FACE_KEY')

if HUGGING_FACE_KEY is None:
    print("WARNING: Hugging Face API key (HUGGING_FACE_KEY) not found in Colab secrets.")

    # Prompt user for input if not found in secrets
    HUGGING_FACE_KEY = input("Please enter your Hugging Face API Key: ")
    if not HUGGING_FACE_KEY:
        print("Hugging Face API Key was not provided. Some models might not load correctly.")
        import sys
        sys.exit()
    else:
        os.environ['HF_TOKEN'] = HUGGING_FACE_KEY
        print("Hugging Face API key received from input.")
else:
    # Set the environment variable for Hugging Face if found in secrets
    os.environ['HF_TOKEN'] = HUGGING_FACE_KEY
    print("Hugging Face API key loaded successfully from secrets.")

Hugging Face API key loaded successfully from secrets.


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch
model_name = "CoDoCGen/CoDoCGen-7B"
model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"

def load_codocgen_model():
    """Loads the CoDocGen model and tokenizer."""
    model = AutoModelForCausalLM.from_pretrained(model_name,
                                                  torch_dtype="auto",
                                                  device_map="auto")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return tokenizer, model

tokenizer, model = load_codocgen_model()
print(f"{model_name} model and tokenizer loaded.")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Qwen/Qwen2.5-Coder-3B-Instruct model and tokenizer loaded.


In [7]:
def generate_documentation(code: str, max_length=512, num_return_sequences=1) -> list:
    """
    Generates documentation for a given code snippet using the CoDoCGen model.

    Args:
        code (str): The code for which to generate documentation.
        max_length (int): The maximum length of the generated documentation.
        num_return_sequences (int): The number of different documentation sequences to generate.

    Returns:
        list: A list of generated documentation strings.
    """

    prompt = f"generate good detailed documentation for this software code: {code}"
    messages = [
      {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
      {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response
    # input_text = f"document code: {code_snippet}"
    # inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)

    # # Generate documentation
    # with torch.no_grad():
    #     outputs = model.generate(
    #         **inputs,
    #         max_length=max_length,
    #         num_return_sequences=num_return_sequences,
    #         no_repeat_ngram_size=2,
    #         early_stopping=True
    #     )

    # generated_docs = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    # return generated_docs

In [9]:
# Test the documentation generation
sample_code = """
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
"""

print("Generating documentation for the following code:")
print(sample_code)

documentation = generate_documentation(sample_code)

print("\nGenerated Documentation:")
print(documentation)

Generating documentation for the following code:

def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)


Generated Documentation:
### Documentation for the `factorial` Function

#### Purpose
The `factorial` function is designed to compute the factorial of a non-negative integer `n`. The factorial of a non-negative integer \( n \) is the product of all positive integers less than or equal to \( n \). It is denoted by \( n! \).

#### Function Signature
```python
def factorial(n):
```

#### Parameters
- **`n`:** A non-negative integer for which the factorial is to be calculated.

#### Returns
- The factorial of the input integer `n`.

#### Example Usage
```python
# Calculate the factorial of 5
result = factorial(5)
print(result)  # Output: 120
```

#### Explanation
The function uses recursion to calculate the factorial. Here's how it works:

1. **Base Case:** If `n` is 0, the function returns 1. This is because the factorial of 0 is defined as 1.
2

## Load and Filter Code Dataset

A function to load a `bigcode/the-stack` dataset and filter it to include only C++ and Python code. The `language` column has the type of laguage

In [58]:
from datasets import load_dataset, interleave_datasets

def load_and_filter_code_dataset(languages:list =None):
    """
    Loads a code dataset and filters it by specified languages.

    Args:
        dataset_name (str): The name of the dataset to load (e.g., "codeparrot/github-code").
        languages (list): A list of programming languages to filter by (e.g., ['C++', 'Python']).
                          If None, no language filtering is applied.

    Returns:
        datasets.Dataset: The filtered dataset.
    """
    dataset_name ="bigcode/the-stack-dedup"
    languages_and_dataset = [
                              {
                                  'language':'python',
                                  'dataset':None
                              },
                              {
                                  'language':'cpp',
                                  'dataset':None
                              }
                            ]

    print(f"Loading dataset: {dataset_name}")
    train_dataset = None

    for i in range(len(languages_and_dataset)):
      language = languages_and_dataset[i]['language']
      print(f"Fetching dataset for '{language}' language")
      # 2. Merge them into one combined stream
      # probabilities=[0.5, 0.5] mixes them evenly (1 Python, 1 C++, 1 Python...)
      languages_and_dataset[i]['dataset'] = load_dataset(dataset_name,
                                       data_dir = f"data/{language}",
                                       split="train",
                                       streaming=True,
                                        token=True)


    return languages_and_dataset

### Test: Loading and Filtering Code

Use the `load_and_filter_code_dataset` function to get only C++ and Python code from the `codeparrot/github-code` dataset.

In [60]:
try:
    cpp_python_dataset = load_and_filter_code_dataset()

    print(next(iter(cpp_python_dataset[0]['dataset'])))
    print(next(iter(cpp_python_dataset[1]['dataset'])))
    # print("\nFirst example from filtered dataset (showing language and a snippet of code):")
    # first_example_filtered = next(iter(cpp_python_dataset))
    # print(f"---\nLanguage: {first_example_filtered.get('lang', 'N/A')}\nCode Snippet: {first_example_filtered.get('content', 'N/A')[:200]}...")

    # The original intent was to get 5 examples, but for streaming datasets,
    # directly indexing or taking len() can be problematic. Iterating explicitly is safer.
    # Let's just confirm the first example for now.

except RuntimeError as e:
    if "Dataset scripts are no longer supported" in str(e):
        print(f"\nError loading dataset: {e}")
        print("\nIt appears the `codeparrot/github-code` dataset cannot be loaded directly via script anymore.")
        print("To fix this, please modify the `load_and_filter_code_dataset` function in cell `CnarbxKBNomR`.")
        print("You might need to specify a `config_name` (e.g., 'all' or 'code_x_m') and potentially use `streaming=True` if the dataset is very large, like so:")
        print("    `dataset = load_dataset(dataset_name, 'all', split=\"train\", streaming=True)`")
        print("Alternatively, you might need to find a different version of the dataset or a more compatible dataset for demonstration purposes.")
    else:
        raise e

Loading dataset: bigcode/the-stack-dedup
Fetching dataset for 'python' language


Resolving data files:   0%|          | 0/144 [00:00<?, ?it/s]

Fetching dataset for 'cpp' language


Resolving data files:   0%|          | 0/110 [00:00<?, ?it/s]

{'hexsha': 'd99a1e98eccb58cbc0c0cef6e9e6702f33461b0e', 'size': 5886, 'ext': 'py', 'lang': 'Python', 'max_stars_repo_path': 'public_data/serializers.py', 'max_stars_repo_name': 'MTES-MCT/sparte', 'max_stars_repo_head_hexsha': '3b8ae6d21da81ca761d64ae9dfe2c8f54487211c', 'max_stars_repo_licenses': ['MIT'], 'max_stars_count': None, 'max_stars_repo_stars_event_min_datetime': None, 'max_stars_repo_stars_event_max_datetime': None, 'max_issues_repo_path': 'public_data/serializers.py', 'max_issues_repo_name': 'MTES-MCT/sparte', 'max_issues_repo_head_hexsha': '3b8ae6d21da81ca761d64ae9dfe2c8f54487211c', 'max_issues_repo_licenses': ['MIT'], 'max_issues_count': 3, 'max_issues_repo_issues_event_min_datetime': '2022-02-10T11:47:58.000Z', 'max_issues_repo_issues_event_max_datetime': '2022-02-23T18:50:24.000Z', 'max_forks_repo_path': 'public_data/serializers.py', 'max_forks_repo_name': 'MTES-MCT/sparte', 'max_forks_repo_head_hexsha': '3b8ae6d21da81ca761d64ae9dfe2c8f54487211c', 'max_forks_repo_licenses'

In [47]:
print("\nFirst example from cpp_python_dataset (using next(iter())):\n")
first_example = next(iter(cpp_python_dataset))
print(first_example)


First example from cpp_python_dataset (using next(iter())):

{'hexsha': 'd66b6e8d1802ed0a290dd994b9af0da47fc99e83', 'size': 475, 'ext': 'abap', 'lang': 'ABAP', 'max_stars_repo_path': 'src/ixml/if_ixml_node_list.intf.abap', 'max_stars_repo_name': 'FreHu/deps', 'max_stars_repo_head_hexsha': 'cace18b54b325d99e4c54293624c1d2811a68ddd', 'max_stars_repo_licenses': ['MIT'], 'max_stars_count': None, 'max_stars_repo_stars_event_min_datetime': None, 'max_stars_repo_stars_event_max_datetime': None, 'max_issues_repo_path': 'src/ixml/if_ixml_node_list.intf.abap', 'max_issues_repo_name': 'FreHu/deps', 'max_issues_repo_head_hexsha': 'cace18b54b325d99e4c54293624c1d2811a68ddd', 'max_issues_repo_licenses': ['MIT'], 'max_issues_count': None, 'max_issues_repo_issues_event_min_datetime': None, 'max_issues_repo_issues_event_max_datetime': None, 'max_forks_repo_path': 'src/ixml/if_ixml_node_list.intf.abap', 'max_forks_repo_name': 'FreHu/deps', 'max_forks_repo_head_hexsha': 'cace18b54b325d99e4c54293624c1d281

### Generate Abstract Syntax Tree (AST) with Tree-sitter

To generate an Abstract Syntax Tree (AST) for code, we can use the `tree-sitter` library. This involves installing the library and specific language parsers.

In [62]:
!pip install tree-sitter-cpp tree-sitter-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 12.9 MB/s eta 0:00:00


In [63]:
from tree_sitter import Language, Parser
import tree_sitter_cpp as tscpp

cpp_language = Language(tscpp.language())

parser = Parser()
parser.language = cpp_language

code = b"""
class Singleton
{
public:
    static Singleton& getInstance();
};
"""

tree = parser.parse(code)

print(tree.root_node)

(translation_unit (class_specifier name: (type_identifier) body: (field_declaration_list (access_specifier) (field_declaration (storage_class_specifier) type: (type_identifier) declarator: (reference_declarator (function_declarator declarator: (field_identifier) parameters: (parameter_list)))))))


### Function generate the AST Given the code
The generate_ast_with_tree_sitter function will use the manually loaded Language objects and generate AST for the given code.
Expect input to be a string or bytes. If string then convert to tbytes before processing with tree sitter.

In [131]:
from tree_sitter import Tree
from typing import Union

def generate_ast_with_tree_sitter(code: Union[str, bytes], language_name: str):
    """
    Generates an Abstract Syntax Tree (AST) for a given code snippet
    using tree-sitter for the specified language.

    Args:
        code (Union[str, bytes]): The code for which to generate the AST. Can be a string or bytestring.
        language_name (str): The name of the programming language (e.g., 'python', 'cpp').

    Returns:
        tree_sitter.Tree: The generated AST.
    """

    from tree_sitter import Language, Parser
    import tree_sitter_cpp as tscpp
    import tree_sitter_python as tspy

    # Check if the language is supported
    if language_name.lower() not in ['cpp', 'c++', 'python']:
      raise ValueError(f"Supported programming languages are 'C++', 'Python'. '{language_name}' is not supported!")

    # Encode string to bytes if necessary
    if isinstance(code, str):
        code = code.encode('utf-8')

    # Load the language dynamically using tree-sitter-languages.
    # This function directly returns a tree_sitter.Language object.
    if language_name.lower() in ['cpp', 'c++']:
      prog_language = Language(tscpp.language())

    elif language_name.lower() == 'python':
      prog_language = Language(tspy.language())

    else:
      raise ValueError(f"Runtime environment error for language {language_name}")


    parser = Parser()
    parser.language = prog_language

    tree = parser.parse(code)
    return tree

def print_ast_tree(tree: Tree, indent=0):
    """
    Helper function to print AST nodes recursively with indentation.
    """
    def _print_node_recursive(node, current_indent):
        print(f"{_get_indent_string(current_indent)}{node.type} [start={node.start_point}, end={node.end_point}]")
        for child in node.children:
            _print_node_recursive(child, current_indent + 1)

    def _get_indent_string(current_indent):
        return '  ' * current_indent

    _print_node_recursive(tree.root_node, indent)

### Test: Generating ASTs

Let's demonstrate `generate_ast_with_tree_sitter` with a Python function and a C++ snippet.

In [133]:
# Example 1: Python Code
python_code = """
def greet(name):
    print(f"Hello, {name}!")
"""

print("Generating AST for Python code:")
python_ast = generate_ast_with_tree_sitter(python_code, 'python')
if python_ast:
    print_ast_tree(python_ast)

print("\n" + "="*50 + "\n")

# Example 2: C++ Code
cpp_code = b"""
#include <iostream>

int main() {
    std::cout << "Hello from C++!" << std::endl;
    return 0;
}
"""

print("Generating AST for C++ code:")
cpp_ast = generate_ast_with_tree_sitter(cpp_code, 'cpp')
if cpp_ast:
    print_ast_tree(cpp_ast)

Generating AST for Python code:
module [start=Point(row=1, column=0), end=Point(row=3, column=0)]
  function_definition [start=Point(row=1, column=0), end=Point(row=2, column=28)]
    def [start=Point(row=1, column=0), end=Point(row=1, column=3)]
    identifier [start=Point(row=1, column=4), end=Point(row=1, column=9)]
    parameters [start=Point(row=1, column=9), end=Point(row=1, column=15)]
      ( [start=Point(row=1, column=9), end=Point(row=1, column=10)]
      identifier [start=Point(row=1, column=10), end=Point(row=1, column=14)]
      ) [start=Point(row=1, column=14), end=Point(row=1, column=15)]
    : [start=Point(row=1, column=15), end=Point(row=1, column=16)]
    block [start=Point(row=2, column=4), end=Point(row=2, column=28)]
      expression_statement [start=Point(row=2, column=4), end=Point(row=2, column=28)]
        call [start=Point(row=2, column=4), end=Point(row=2, column=28)]
          identifier [start=Point(row=2, column=4), end=Point(row=2, column=9)]
          ar